# 第02课：矩阵 —— 神经网络的积木

> **前置要求**：完成第01课（向量基础）

## 这节课你将理解

- 矩阵是什么？跟向量有什么关系？
- 矩阵乘法怎么算？为什么 AI 离不开它？
- 神经网络「一层」到底在算什么？（答案：一次矩阵乘法 + 一次加法）
- 转置是什么？为什么 Attention 公式里有个 K.T？

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys; sys.path.append('..')
from utils import zh_font

print('准备好了！')

---
# 一、矩阵是什么？

## 1.1 从表格说起

上节课我们用**一个向量**描述一个人。如果有 3 个人呢？

| 人 | 身高 | 体重 | 年龄 |
|----|------|------|------|
| 小明 | 175 | 70 | 25 |
| 小红 | 165 | 55 | 23 |
| 小刚 | 180 | 80 | 28 |

把这张表格的数字部分抠出来，就是一个**矩阵**：

```
[[175, 70, 25],
 [165, 55, 23],
 [180, 80, 28]]
```

> 💡 **矩阵 = 多个向量摞在一起组成的二维数字表格。**
>
> - **每一行** = 一个样本（一个人/一个词/一张图）
> - **每一列** = 一个特征（身高/体重/年龄）

## 1.2 形状怎么看

矩阵的形状写作 **(行数, 列数)**。

上面的矩阵：3 行 3 列 → 形状是 `(3, 3)`。

AI 里最常见的情况：
- 输入数据 `X` 形状 = `(batch_size, feature_dim)` → 有多少个样本 × 每个样本多少维

In [ ]:
# 创建一个矩阵：3个词向量，每个4维
# 每一行是一个词的向量
X = np.array([
    [0.2, 0.8, -0.1, 0.5],   # king
    [0.3, 0.7, -0.2, 0.6],   # queen
    [0.1, 0.6, -0.3, 0.4],   # man
])

print('矩阵 X:')
print(X)
print()
print(f'形状: {X.shape}')   # (3, 4) → 3个词，每个4维

In [ ]:
# 取行和取列

print('第0行（king的向量）:', X[0])       # 取出一行 → 得到一个向量
print('第1列（所有词的第2个特征）:', X[:, 1])  # : 表示所有行

---
# 二、矩阵乘法 —— 神经网络的核心运算

## 2.1 先回顾点积

上节课学过：
```
[1, 2, 3] · [4, 5, 6] = 1×4 + 2×5 + 3×6 = 32
```
一行 × 一列 → **一个数字**。

## 2.2 矩阵乘法 = 一大批点积

矩阵乘法就是：**A 的每一行，分别跟 B 的每一列做点积**，把结果填进新矩阵。

举个小例子：

```
A = [[1, 2],     B = [[5, 6],
     [3, 4]]          [7, 8]]

C = A @ B

C[0,0] = A的第0行 · B的第0列 = [1,2] · [5,7] = 1×5 + 2×7 = 19
C[0,1] = A的第0行 · B的第1列 = [1,2] · [6,8] = 1×6 + 2×8 = 22
C[1,0] = A的第1行 · B的第0列 = [3,4] · [5,7] = 3×5 + 4×7 = 43
C[1,1] = A的第1行 · B的第1列 = [3,4] · [6,8] = 3×6 + 4×8 = 50

C = [[19, 22],
     [43, 50]]
```

> 💡 记不住也没关系，numpy 一个 `@` 就帮你算完了。关键是理解**形状规则**。

In [ ]:
# 验证上面的例子

A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

C = A @ B      # @ 就是矩阵乘法

print('A @ B =')
print(C)

# 手动验证第一个元素
print(f'\n验证 C[0,0]: {A[0]} · {B[:, 0]} = {A[0] @ B[:, 0]}')

## 2.3 形状规则（必须记住！）

```
(m, k) @ (k, n) → (m, n)
       ↑↑↑
    中间这个 k 必须相等！
```

打个比方：

- A 是一张 **m行k列** 的表
- B 是一张 **k行n列** 的表
- A 的「列数」必须等于 B 的「行数」，就像**插头和插座**要匹配
- 结果是 **m行n列** 的新表

```
  A           B         结果
(3, 4)  @  (4, 2)  →  (3, 2)
    ↑        ↑
    └── 4 = 4 ──┘  ← 匹配！
```

In [ ]:
# 验证形状规则

A = np.random.randn(3, 4)   # 3行4列
B = np.random.randn(4, 2)   # 4行2列  （4=4，匹配）

C = A @ B
print(f'{A.shape} @ {B.shape} → {C.shape}')   # (3,4) @ (4,2) → (3,2)

In [ ]:
# 不匹配会报错

A = np.random.randn(3, 4)   # 3行4列
B = np.random.randn(5, 2)   # 5行2列  （4 ≠ 5，不匹配）

try:
    C = A @ B
except ValueError as e:
    print(f'报错了！{e}')
    print(f'因为 A 的列数({A.shape[1]}) ≠ B 的行数({B.shape[0]})')

---
# 三、神经网络的「一层」到底在算什么？

## 3.1 全连接层 = 一次矩阵乘法 + 一次加法

神经网络最基本的一层叫**全连接层**（Linear Layer），公式极其简单：

$$
Y = X \times W + b
$$

翻译成人话：

| 符号 | 含义 | 形状 | 比喻 |
|------|------|------|------|
| X | 输入数据 | (batch, in_dim) | 一批学生的考试成绩 |
| W | 权重矩阵 | (in_dim, out_dim) | 老师给各科的「加权系数」 |
| b | 偏置 | (out_dim,) | 基础分 |
| Y | 输出 | (batch, out_dim) | 加权后的总分 |

形状验证：
```
(3, 4) @ (4, 2) → (3, 2)
 3个样本  变换矩阵  3个样本
 每个4维            每个变成2维
```

**就这么简单！GPT 里说的「175B 参数」，绝大部分就是这些 W 矩阵里的数字。**

In [ ]:
# 模拟一层全连接网络

# 输入：3个词，每个4维
X = np.array([
    [0.2, 0.8, -0.1, 0.5],   # king
    [0.3, 0.7, -0.2, 0.6],   # queen
    [0.1, 0.6, -0.3, 0.4],   # man
])

# 权重：把4维变成2维（这就是「参数」，训练时会不断调整）
np.random.seed(42)   # 固定随机种子，让每次结果一样
W = np.random.randn(4, 2) * 0.3   # (4, 2)

# 偏置：每个输出维度一个
b = np.array([0.1, -0.1])

# 前向传播！
Y = X @ W + b

print(f'输入 X:  形状 {X.shape}')    # (3, 4)
print(f'权重 W:  形状 {W.shape}')    # (4, 2)
print(f'偏置 b:  形状 {b.shape}')    # (2,)
print(f'输出 Y:  形状 {Y.shape}')    # (3, 2) ← 3个词，每个变成2维了
print()
print('输出 Y =')
print(Y)

**发生了什么？**

3 个 4 维的词向量，经过一次矩阵乘法，变成了 3 个 2 维的向量。

这就是**降维** —— 从 4 维空间投影到 2 维空间。

反过来也行：如果 W 的形状是 (4, 8)，就会从 4 维变成 8 维 —— **升维**。

GPT 的 FFN 层就是：先升维（比如 4096 → 16384），再降回去（16384 → 4096）。

In [ ]:
# 💡 试试改 W 的形状，看输出怎么变

W_big = np.random.randn(4, 8) * 0.3     # 4维 → 8维（升维）
Y_big = X @ W_big + np.zeros(8)          # 偏置也要跟着变

print(f'升维: {X.shape} @ {W_big.shape} → {Y_big.shape}')

W_small = np.random.randn(4, 1) * 0.3   # 4维 → 1维（压到一个数字）
Y_small = X @ W_small

print(f'降到1维: {X.shape} @ {W_small.shape} → {Y_small.shape}')

---
# 四、三种「乘法」的区别

这是初学者**最容易搞混**的地方。分清这三种，后面学 Attention 就不会晕。

| 操作 | 写法 | 规则 | 结果形状 | AI 用途 |
|------|------|------|---------|--------|
| **逐元素乘** | `A * B` | 对应位置相乘 | 跟输入一样 | 门控(LSTM遗忘门)、mask |
| **矩阵乘** | `A @ B` | 行×列做点积 | (m,k)@(k,n)→(m,n) | **神经网络层** |
| **标量乘** | `3 * A` | 每个元素×同一个数 | 跟输入一样 | 学习率缩放 |

In [ ]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

print('=== 逐元素乘 A * B ===')
print(A * B)
print('规则: 1×5=5, 2×6=12, 3×7=21, 4×8=32')

print()
print('=== 矩阵乘 A @ B ===')
print(A @ B)
print('规则: 第0行·第0列=1×5+2×7=19, ...')

print()
print('=== 标量乘 3 * A ===')
print(3 * A)
print('规则: 每个元素×3')

---
# 五、转置 —— Attention 里的 K.T

## 5.1 转置 = 行变列、列变行

就是把矩阵「翻个面」：原来的第 i 行变成第 i 列。

```
原矩阵 (2, 3):         转置后 (3, 2):
[[1, 2, 3],             [[1, 4],
 [4, 5, 6]]              [2, 5],
                          [3, 6]]
```

形状：`(m, n)` → `(n, m)`

## 5.2 为什么 AI 里需要它？

Attention 的公式你以后会学到：
```
score = Q @ K.T
```

这里 Q 和 K 的形状都是 `(seq_len, dim)`。

直接 Q @ K 不行 → `(seq, dim) @ (seq, dim)` → 中间的 dim ≠ seq，不匹配！

把 K 转置后 → `(seq, dim) @ (dim, seq)` → 中间 dim = dim，匹配了！

结果形状是 `(seq, seq)` → 每个词对其他每个词的「注意力分数」。

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])

print(f'原矩阵 形状: {M.shape}')
print(M)

print(f'\n转置 形状: {M.T.shape}')   # .T 就是转置
print(M.T)

In [ ]:
# 模拟 Attention 的形状逻辑

seq_len = 3   # 句子长度（比如 "猫 吃 鱼" 三个词）
dim = 4       # 每个词的向量维度

Q = np.random.randn(seq_len, dim)   # Query
K = np.random.randn(seq_len, dim)   # Key

# 直接乘不行：
print(f'Q 形状: {Q.shape},  K 形状: {K.shape}')
print(f'Q @ K 行不通 → {Q.shape} @ {K.shape} 中间 {Q.shape[1]} ≠ {K.shape[0]}')

# 转置 K 后就行了：
score = Q @ K.T
print(f'\nQ @ K.T → {Q.shape} @ {K.T.shape} = {score.shape}')
print(f'\n注意力分数矩阵（每个词对每个词的相关性）:')
print(np.round(score, 2))

---
# 六、实战：手写一个全连接层

在 PyTorch 里写 `nn.Linear(4, 2)` 就是创建一层。

但它内部到底做了什么？就是下面这么几行：

In [ ]:
class Linear:
    """
    手写的全连接层，跟 PyTorch 的 nn.Linear 做的事完全一样。
    
    初始化时随机生成权重 W 和偏置 b，
    调用时计算 Y = X @ W + b。
    """
    def __init__(self, in_dim, out_dim):
        # He 初始化：乘以 √(2/n) 让初始值不会太大也不会太小
        self.W = np.random.randn(in_dim, out_dim) * np.sqrt(2.0 / in_dim)
        self.b = np.zeros(out_dim)   # 偏置初始化为 0
    
    def __call__(self, x):
        # __call__ 让类的实例可以像函数一样被调用：layer(x)
        return x @ self.W + self.b

# 用法跟 PyTorch 一模一样
layer = Linear(4, 2)
print(f'权重形状: {layer.W.shape}')   # (4, 2)
print(f'偏置形状: {layer.b.shape}')   # (2,)')

In [ ]:
# 用我们的 Linear 搭一个 2 层网络

layer1 = Linear(4, 8)    # 4维 → 8维
layer2 = Linear(8, 2)    # 8维 → 2维

h = layer1(X)             # 第1层：(3,4) → (3,8)
out = layer2(h)           # 第2层：(3,8) → (3,2)

print(f'输入:     {X.shape}')
print(f'第1层后:  {h.shape}')
print(f'第2层后:  {out.shape}')
print()
print('最终输出:')
print(np.round(out, 3))

---
# 七、可视化：矩阵乘法 = 几何变换

矩阵乘法除了「处理数据」，还有一个很美的几何意义：

**用一个矩阵乘一批点 = 对这些点做「变换」（旋转、缩放、扭曲等）。**

下面我们用一个旋转矩阵，把几个点旋转 45°。

In [ ]:
# 旋转矩阵（旋转45度）
theta = np.pi / 4   # 45° 换算成弧度

# 这个 2×2 矩阵能把任何 2D 点旋转 theta 角
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

print('旋转矩阵 R =')
print(np.round(R, 3))

In [ ]:
# 一些 2D 点
points = np.array([[1, 0], [1, 1], [0, 1], [0.5, 0.5]])

# 矩阵乘法 = 旋转
# 每个点是一行，R.T 让旋转方向正确
rotated = points @ R.T

# 画图
fig, ax = plt.subplots(figsize=(6, 6))

for i, label in enumerate(['A', 'B', 'C', 'D']):
    p, q = points[i], rotated[i]
    
    # 原始点（蓝色圆）
    ax.plot(*p, 'bo', markersize=10)
    ax.text(p[0]+0.05, p[1]+0.05, label, fontsize=12, color='blue')
    
    # 旋转后的点（红色方块）
    ax.plot(*q, 'rs', markersize=10)
    ax.text(q[0]+0.05, q[1]+0.05, f"{label}'", fontsize=12, color='red')
    
    # 连线
    ax.plot([p[0], q[0]], [p[1], q[1]], 'gray', alpha=0.3, linestyle='--')

ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlim(-1.2, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.set_title('矩阵乘法 = 旋转45°', fontsize=14, fontproperties=zh_font)
plt.show()

print('蓝色 = 原始点，红色 = 旋转后的点')
print('矩阵乘法把所有点同时旋转了！')

---
# 八、本节总结

| 概念 | 一句话理解 | AI 用途 |
|-----|----------|--------|
| **矩阵** | 多个向量摞成的二维表格 | 一批样本、权重参数 |
| **矩阵乘法** | A的每行 × B的每列 做点积 | **神经网络层的核心** |
| **形状规则** | (m,k) @ (k,n) → (m,n)，中间k必须相等 | 检查维度是否正确 |
| **全连接层** | Y = X @ W + b | GPT 参数的主体 |
| **逐元素乘** | A * B，对应位置相乘 | 门控、mask |
| **转置** | 行变列、列变行 | Attention 中的 K.T |

### 在 GPT 一个 block 里的矩阵乘法

```python
# 这就是 Transformer 一个 block 的核心运算（简化版）
QKV = x @ W_qkv           # 投影出 Q, K, V
score = Q @ K.T            # 注意力分数（下节课详讲）
attn_out = score @ V       # 加权求和
out = attn_out @ W_proj    # 输出投影
ffn = out @ W_up           # FFN 升维
final = ffn @ W_down       # FFN 降维
```

**全部是矩阵乘法。** 这就是为什么 GPU（擅长并行矩阵运算）对 AI 这么重要。

---
# 九、练习

## 题1：形状推理

创建 X(4×3) 和 W(3×5)，算 X@W 并打印形状。

In [ ]:
# TODO: 创建两个矩阵并相乘


## 题2：3 层网络

用上面的 `Linear` 类搭一个 3 层网络：10→20→20→5，传入一个 (4, 10) 的输入。

In [ ]:
# TODO: 搭 3 层网络


## 题3：转置乘法律

验证：`(A @ B).T` 等于 `B.T @ A.T`

提示：用 `np.allclose()` 比较两个矩阵是否相等（浮点数不能直接用 `==`）。

In [ ]:
# TODO: 验证 (A @ B).T == B.T @ A.T


---
## 答案（卡住了再看）

In [ ]:
# === 题1 ===
X1 = np.random.randn(4, 3)
W1 = np.random.randn(3, 5)
Y1 = X1 @ W1
print(f'题1: {X1.shape} @ {W1.shape} → {Y1.shape}')   # (4,5)

# === 题2 ===
l1 = Linear(10, 20)
l2 = Linear(20, 20)
l3 = Linear(20, 5)

inp = np.random.randn(4, 10)
o1 = l1(inp)
o2 = l2(o1)
o3 = l3(o2)
print(f'题2: {inp.shape} → {o1.shape} → {o2.shape} → {o3.shape}')

# === 题3 ===
A3 = np.random.randn(3, 4)
B3 = np.random.randn(4, 5)
left  = (A3 @ B3).T       # 先乘再转
right = B3.T @ A3.T       # 各自转再乘（注意顺序反了）
print(f'题3: (A@B).T == B.T@A.T? {np.allclose(left, right)}')

---
🎉 **第02课完成！**

你现在掌握了 AI 里最核心的数学操作 —— 矩阵乘法。

下一课（第03课）我们会讲**线性变换**：矩阵乘法到底在「变换」什么？换一组「坐标系」看世界是什么意思？